# 03 - Inference with the packaged model

Shows the three ways to use the trained artefact: the joblib pipeline, the MLflow pyfunc model, and the REST API.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.width", 140)

In [2]:
from heart_risk.predict import load_model, predict_records
from heart_risk import config
model = load_model(str(config.MODEL_PATH)); model

Pipeline(steps=[('preprocess',
                 Pipeline(steps=[('clinical', ClinicalFeatureBuilder()),
                                 ('columns',
                                  ColumnTransformer(transformers=[('num',
                                                                   Pipeline(steps=[('impute',
                                                                                    SimpleImputer(strategy='median')),
                                                                                   ('scale',
                                                                                    StandardScaler())]),
                                                                   ['age',
                                                                    'trestbps',
                                                                    'chol',
                                                                    'thalach',
                                                                    'oldpeak',
                                                                    'ca',
                                                                    'hr_reserve_ratio']),
                                                                  ('bin',
                                                                   Pipeline(steps=[('impute',
                                                                                    SimpleImputer(strategy=...
                                                                    'exang']),
                                                                  ('cat',
                                                                   Pipeline(steps=[('impute',
                                                                                    SimpleImputer(strategy='most_frequent')),
                                                                                   ('onehot',
                                                                                    OneHotEncoder(categories=[[1,
                                                                                                               2,
                                                                                                               3,
                                                                                                               4],
                                                                                                              [0,
                                                                                                               1,
                                                                                                               2],
                                                                                                              [1,
                                                                                                               2,
                                                                                                               3],
                                                                                                              [3,
                                                                                                               6,
                                                                                                               7]],
                                                                                                  handle_unknown='ignore',
                                                                                                  sparse_output=False))]),
                                                                   ['cp',
                                                                    'restecg',
                                                                    'slope',
                                                                    'thal'])]))])),
                ('mo

## 1. joblib pipeline (raw features in, probability out)

In [3]:
import json
patient = json.loads((ROOT / 'sample_request.json').read_text())
predict_records([patient], model=model)

[{'prediction': 1,
  'label': 'heart_disease',
  'confidence': 0.9767,
  'risk_probability': 0.9767}]

In [4]:
# missing optional fields are imputed by the pipeline
partial = {k: v for k, v in patient.items() if k not in ('ca', 'thal')}
predict_records([partial, {**patient, 'cp': 3, 'exang': 0, 'oldpeak': 0.0, 'thalach': 172}], model=model)

[{'prediction': 1,
  'label': 'heart_disease',
  'confidence': 0.7928,
  'risk_probability': 0.7928},
 {'prediction': 1,
  'label': 'heart_disease',
  'confidence': 0.7111,
  'risk_probability': 0.7111}]

## 2. MLflow model format

In [5]:
import mlflow.pyfunc
pyfunc = mlflow.pyfunc.load_model(str(config.MLFLOW_MODEL_DIR))
print(pyfunc.metadata.get_input_schema())
# cast to the column dtypes recorded in the MLflow model signature
dtypes = pd.read_csv(config.CLEAN_DATA_PATH)[config.FEATURES].dtypes.to_dict()
pyfunc.predict(pd.DataFrame([patient])[config.FEATURES].astype(dtypes))

['age': double (required), 'trestbps': double (required), 'chol': double (required), 'thalach': double (required), 'oldpeak': double (required), 'ca': double (optional), 'sex': long (required), 'fbs': long (required), 'exang': long (required), 'cp': double (required), 'restecg': double (required), 'slope': double (required), 'thal': double (optional)]


array([1], dtype=int64)

## 3. Batch scoring of the clean dataset (sanity check)

In [6]:
from heart_risk import data, evaluate
df = data.load_clean()
prob = model.predict_proba(df[config.FEATURES])[:, 1]
pd.Series(evaluate.classification_metrics(df['target'], prob)).round(4)

accuracy         0.8581
precision        0.8529
recall           0.8345
f1               0.8436
roc_auc          0.9323
avg_precision    0.9241
dtype: float64

## 4. REST API
Start the service with `uvicorn api.main:app --port 8000` (or `docker run -p 8000:8000 heart-risk-api:1.0.0`) and run the cell below.

In [7]:
import requests
try:
    r = requests.post('http://127.0.0.1:8000/predict', json=patient, timeout=5)
    print(r.status_code, r.json())
except requests.ConnectionError:
    print('API not running - start it first')

200 {'prediction': 1, 'label': 'heart_disease', 'confidence': 0.9767, 'risk_probability': 0.9767, 'model_version': '1.0.0', 'request_id': 'c5cebc2a992e'}
